# Therapeutic Optimization — Colab Runner

This notebook is intentionally lightweight. All reusable logic lives in `src/therapeutic_optimization/`.

Pipeline: **T1 → UP1 → T2 → S1 → R1 → UB2 → R2**.

Before publishing, replace the placeholder GitHub URL below with the real repository URL.


## 0. Runtime

Use a **GPU runtime**. EUP uses ESM2-3B and this implementation intentionally fails rather than silently falling back to a very slow CPU path.


In [1]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/juliaevizza/therapeutic_optimization.git"
REPO_DIR = Path("/content/therapeutic_optimization")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repository ready:", REPO_DIR)


Repository ready: /content/therapeutic_optimization


## 1. Install dependencies

EUP assets and the ESM2 cache stay in the Colab runtime, not in your Drive/repository. ColabFold is installed separately because its JAX/CUDA packages are runtime-specific.


In [2]:
# Run this after REPO_URL is configured and the repository has been cloned.
if REPO_DIR.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[eup]"], check=True)

    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)

    if shutil.which("colabfold_batch") is None:
        print("colabfold_batch not found. Installing the current CUDA 12 ColabFold stack...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "colabfold[alphafold,openmm]", "jax[cuda12]", "openmm[cuda12]"
        ], check=True)

    subprocess.run([sys.executable, str(REPO_DIR / "scripts" / "check_environment.py")], check=False)


colabfold_batch not found. Installing the current CUDA 12 ColabFold stack...


## 2. Choose where run outputs live

Set `SAVE_TO_DRIVE = True` if you want the run to survive Colab shutdown. The package code still runs from GitHub; only `storage/` outputs go into this run directory.


In [3]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/therapeutic_optimization_run")
else:
    RUN_ROOT = Path("/content/therapeutic_optimization_run")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Run root:", RUN_ROOT)


Mounted at /content/drive
Run root: /content/drive/MyDrive/therapeutic_optimization_run


## 3. Hyperparameters

`single` changes one selected lysine at a time. `combinatorial` generates orders 1 through `MAX_COMBINATION_ORDER` using every replacement amino acid listed in `REPLACEMENT_AAS`.


In [4]:
UBI_THRESHOLD = 0.40

MUTATION_MODE = "single"            # "single" or "combinatorial"
REPLACEMENT_AAS = ("A",)            # e.g. ("A", "R", "Q")
MAX_COMBINATION_ORDER = 2
MAX_VARIANTS = 5000

# S1 structural-preservation heuristic gates
GLOBAL_CA_RMSD_MAX = 1.0
LOCAL_MEAN_CA_DISPLACEMENT_MAX = 1.5
MUTATION_CA_DISPLACEMENT_MAX = 2.0
CONTACT_CHANGE_FRACTION_MAX = 0.10
MIN_MEAN_PLDDT = 70.0


## 4. User input


In [7]:
PROTEIN_ID = "my_protein"
WT_SEQUENCE = """PTAPPYDSLLVFDYEGGSGSGSGASRLNFGDDIPSALRIAKKKRWNSIEERRIHQESELHSYLSRLIAAERERELEECQRNHEGDEDDSHVRAQQACIEAKHDKYMADMDELFSQVDEKRKKRDIPDYLCGKISFELMREPCITPSGITYDRKDIEEHLQRVGHFDPVTRSPLTQEQLIPNLAMKEVIDAFISENGWVEDY"""


## 5. Build the workflow


In [14]:
import sys
if 'therapeutic_optimization' in sys.modules:
    del sys.modules['therapeutic_optimization']
if "/content/therapeutic_optimization/src" not in sys.path:
    sys.path.insert(0, "/content/therapeutic_optimization/src")

from therapeutic_optimization.config import (
    MutationConfig,
    PredictorConfig,
    StructuralThresholds,
    WorkflowConfig,
)
from therapeutic_optimization.workflow import OptimizationWorkflow
from pathlib import Path

config = WorkflowConfig(
    mutation=MutationConfig(
        threshold=UBI_THRESHOLD,
        mode=MUTATION_MODE,
        replacement_aas=REPLACEMENT_AAS,
        max_combination_order=MAX_COMBINATION_ORDER,
        max_variants=MAX_VARIANTS,
    ),
    ubiquitination=PredictorConfig(
        name="eup",
        threshold=UBI_THRESHOLD,
        eup_repo_dir=Path("/content/external/EUP"),
        model_cache_dir=Path("/content/huggingface"),
    ),
    structural_thresholds=StructuralThresholds(
        global_ca_rmsd_max=GLOBAL_CA_RMSD_MAX,
        local_mean_ca_displacement_max=LOCAL_MEAN_CA_DISPLACEMENT_MAX,
        mutation_ca_displacement_max=MUTATION_CA_DISPLACEMENT_MAX,
        contact_change_fraction_max=CONTACT_CHANGE_FRACTION_MAX,
        min_mean_plddt=MIN_MEAN_PLDDT,
    ),
)

workflow = OptimizationWorkflow(RUN_ROOT, config)

## T1 — input → WT FASTA


In [15]:
t1 = workflow.T1(WT_SEQUENCE, PROTEIN_ID)
t1


{'protein_id': 'my_protein',
 'sequence_length': 201,
 'wt_fasta': '/content/drive/MyDrive/therapeutic_optimization_run/storage/inputs/wt_input.fasta',
 'created_at_utc': '2026-08-31T19:12:23.271061+00:00'}

## UP1 — WT ubiquitination prediction


In [16]:
up1 = workflow.UP1()
display(up1)


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/58.0k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/582 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t36_3B_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,variant_id,protein_id,predictor,lysine_position,site,sequence_context,probability,threshold,is_positive
0,WT,my_protein,EUP,153,K153,ITPSGITYDRKDIEEHLQRVG,0.609378,0.4,True
1,WT,my_protein,EUP,119,K119,MDELFSQVDEKRKKRDIPDYL,0.547615,0.4,True
2,WT,my_protein,EUP,185,K185,QEQLIPNLAMKEVIDAFISEN,0.524864,0.4,True
3,WT,my_protein,EUP,101,K101,VRAQQACIEAKHDKYMADMDE,0.499223,0.4,True
4,WT,my_protein,EUP,104,K104,QQACIEAKHDKYMADMDELFS,0.460844,0.4,True
5,WT,my_protein,EUP,132,K132,KRDIPDYLCGKISFELMREPC,0.400665,0.4,True
6,WT,my_protein,EUP,122,K122,LFSQVDEKRKKRDIPDYLCGK,0.332685,0.4,False
7,WT,my_protein,EUP,43,K43,IPSALRIAKKKRWNSIEERRI,0.302615,0.4,False
8,WT,my_protein,EUP,41,K41,DDIPSALRIAKKKRWNSIEER,0.288181,0.4,False
9,WT,my_protein,EUP,42,K42,DIPSALRIAKKKRWNSIEERR,0.245417,0.4,False


## T2 — generate mutant FASTAs


In [17]:
t2 = workflow.T2(up1)
print(f"Generated {len(t2)} mutant(s).")
display(t2)


Generated 6 mutant(s).


,variant_id,mutation_spec,mutation_count,source_sites,replacement_aas,sequence_length,fasta_path,status,error
0,K101A,K101A,1,K101,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
1,K104A,K104A,1,K104,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
2,K119A,K119A,1,K119,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
3,K132A,K132A,1,K132,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
4,K153A,K153A,1,K153,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
5,K185A,K185A,1,K185,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None


## S1 — structure prediction + preservation screen

This is the expensive structure stage. Every T2 mutant is predicted and compared with WT.


In [18]:
s1_metrics, s1_conserved = workflow.S1(t2, predict_structures=True)
print(f"Structurally conserved: {len(s1_conserved)} / {len(s1_metrics)}")
display(s1_metrics)
display(s1_conserved)


Structurally conserved: 2 / 6


,variant_id,mutation_spec,mutation_count,source_sites,replacement_aas,sequence_length,fasta_path,status,error,global_ca_rmsd,...,mean_centroid_distance_change,mean_residue_volume_change,per_residue_csv,displacement_plot,wt_structure,mutant_structure,analysis_status,analysis_error,structure_pass,structural_preservation_score
0,K101A,K101A,1,K101,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,6.699305,...,0.118797,-68.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.670432
1,K104A,K104A,1,K104,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.685948,...,0.200159,-68.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.743423
2,K119A,K119A,1,K119,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.722661,...,0.029973,-68.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.855787
3,K132A,K132A,1,K132,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.732838,...,0.062896,-68.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.835882
4,K153A,K153A,1,K153,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,6.857941,...,0.228384,-68.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.599181
5,K185A,K185A,1,K185,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.751329,...,-0.347354,-68.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.643743


,variant_id,mutation_spec,mutation_count,source_sites,replacement_aas,sequence_length,fasta_path,status,error,global_ca_rmsd,...,mean_centroid_distance_change,mean_residue_volume_change,per_residue_csv,displacement_plot,wt_structure,mutant_structure,analysis_status,analysis_error,structure_pass,structural_preservation_score
2,K119A,K119A,1,K119,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.722661,...,0.029973,-68.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.855787
3,K132A,K132A,1,K132,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.732838,...,0.062896,-68.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.835882


## R1 — record structural dropouts


In [19]:
r1 = workflow.R1(t2, s1_metrics)
display(r1)


,variant_id,mutation_spec,mutation_count,source_sites,replacement_aas,sequence_length,fasta_path,status,error,analysis_status,analysis_error,structure_pass,structural_preservation_score,global_ca_rmsd,local_mean_ca_displacement,mutation_ca_displacement_max,global_contact_change_fraction,mutant_mean_plddt,dropped_after_S1,R1_status
0,K119A,K119A,1,K119,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.855787,0.722661,0.235497,0.264932,0.008940,82.969950,False,ADVANCE_TO_UB2
1,K132A,K132A,1,K132,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.835882,0.732838,0.276610,0.518821,0.011465,82.506169,False,ADVANCE_TO_UB2
2,K104A,K104A,1,K104,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,False,0.743423,1.685948,0.290966,0.297565,0.017744,82.018806,True,DROPPED_STRUCTURAL
3,K101A,K101A,1,K101,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,False,0.670432,6.699305,0.605824,0.533762,0.027604,82.555771,True,DROPPED_STRUCTURAL
4,K185A,K185A,1,K185,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,False,0.643743,1.751329,1.091173,1.080999,0.018963,82.317114,True,DROPPED_STRUCTURAL
5,K153A,K153A,1,K153,A,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,False,0.599181,6.857941,1.116704,1.122840,0.033750,81.759900,True,DROPPED_STRUCTURAL


## UB2 — rerun ubiquitination prediction on S1 survivors


In [20]:
ub2 = workflow.UB2(s1_conserved)
display(ub2)


,variant_id,protein_id,predictor,lysine_position,site,sequence_context,probability,threshold,is_positive
0,K119A,K119A,EUP,153,K153,ITPSGITYDRKDIEEHLQRVG,0.586457,0.4,True
1,K119A,K119A,EUP,185,K185,QEQLIPNLAMKEVIDAFISEN,0.493597,0.4,True
2,K119A,K119A,EUP,101,K101,VRAQQACIEAKHDKYMADMDE,0.444029,0.4,True
3,K119A,K119A,EUP,104,K104,QQACIEAKHDKYMADMDELFS,0.436900,0.4,True
4,K119A,K119A,EUP,132,K132,KRDIPDYLCGKISFELMREPC,0.374480,0.4,False
5,K119A,K119A,EUP,122,K122,LFSQVDEARKKRDIPDYLCGK,0.328575,0.4,False
6,K119A,K119A,EUP,43,K43,IPSALRIAKKKRWNSIEERRI,0.288305,0.4,False
7,K119A,K119A,EUP,41,K41,DDIPSALRIAKKKRWNSIEER,0.266816,0.4,False
8,K119A,K119A,EUP,42,K42,DIPSALRIAKKKRWNSIEERR,0.237676,0.4,False
9,K119A,K119A,EUP,121,K121,ELFSQVDEARKKRDIPDYLCG,0.232867,0.4,False


## R2 — final ranking


In [21]:
r2_all, optimized, needs_more = workflow.R2(up1, ub2, s1_conserved)

print("OPTIMIZED")
display(optimized)

print("NEEDS FURTHER OPTIMIZATION")
display(needs_more)

print("ALL RANKED")
display(r2_all)


OPTIMIZED


,final_rank,variant_id,mutation_spec,mutation_count,structural_preservation_score,wt_positive_count,mutant_positive_count,targeted_wt_probability_sum,wt_positive_burden,mutant_positive_burden,ubiquitination_burden_reduction,ubiquitination_burden_reduction_fraction,removed_wt_sites,remaining_wt_positive_sites,new_positive_sites,new_positive_count,R2_group


NEEDS FURTHER OPTIMIZATION


,final_rank,variant_id,mutation_spec,mutation_count,structural_preservation_score,wt_positive_count,mutant_positive_count,targeted_wt_probability_sum,wt_positive_burden,mutant_positive_burden,ubiquitination_burden_reduction,ubiquitination_burden_reduction_fraction,removed_wt_sites,remaining_wt_positive_sites,new_positive_sites,new_positive_count,R2_group
0,1,K119A,K119A,1,0.855787,6,4,0.547615,3.042589,1.960983,1.081606,0.355489,K119;K132,K101;K104;K153;K185,,0,needs_further_optimization
1,2,K132A,K132A,1,0.835882,6,5,0.400665,3.042589,2.641035,0.401554,0.131978,K132,K101;K104;K119;K153;K185,,0,needs_further_optimization


ALL RANKED


,final_rank,variant_id,mutation_spec,mutation_count,structural_preservation_score,wt_positive_count,mutant_positive_count,targeted_wt_probability_sum,wt_positive_burden,mutant_positive_burden,ubiquitination_burden_reduction,ubiquitination_burden_reduction_fraction,removed_wt_sites,remaining_wt_positive_sites,new_positive_sites,new_positive_count,R2_group
0,1,K119A,K119A,1,0.855787,6,4,0.547615,3.042589,1.960983,1.081606,0.355489,K119;K132,K101;K104;K153;K185,,0,needs_further_optimization
1,2,K132A,K132A,1,0.835882,6,5,0.400665,3.042589,2.641035,0.401554,0.131978,K132,K101;K104;K119;K153;K185,,0,needs_further_optimization


## 6. Inspect outputs


In [23]:
print("Tables:")
for path in sorted((RUN_ROOT / "storage" / "tables").glob("*.csv")):
    print(" -", path)

print("Mutant FASTAs:", len(list((RUN_ROOT / "storage" / "mutants" / "fastas").glob("*.fasta"))))
print("WT structure directory:", RUN_ROOT / "storage" / "structures" / "wt")
print("Mutant structure directory:", RUN_ROOT / "storage" / "structures" / "mutants")


Tables:
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/R1_structural_screen.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/R2_all_ranked.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/R2_needs_further_optimization.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/R2_optimized.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/S1_structural_metrics.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/S1_structurally_conserved.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/T2_mutation_manifest.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/UB2_mutant_ubiquitination.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/UP1_wt_ubiquitination.csv
Mutant FASTAs: 6
WT structure directory: /content/drive/MyDrive/therapeutic_optimization_run/storage/structures/wt
Mutant structure